In [1]:
%env HTTPS_PROXY=socks5h://localhost:1080

env: HTTPS_PROXY=socks5h://localhost:1080


In [2]:
!echo $HTTPS_PROXY

socks5h://localhost:1080


In [4]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "unsloth/Meta-Llama-3.1-70B"

# 1. Загрузка токенизатора
# Пытаемся загрузить родной, если HF ругнется на доступ, используем публичный Instruct
try:
    print(f"Loading tokenizer for {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
except Exception as e:
    print(f"Native tokenizer access denied, using public fallback...")
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Загрузка модели
# device_map="auto" автоматически раскидает слои по двум A100 (нужно ~140-150 ГБ VRAM)
print(f"Loading model {model_name} in bfloat16...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

print("Done! Model is distributed across GPUs:")
for i, device in enumerate(set(model.hf_device_map.values())):
    print(f"GPU {device}")

# Тестовая проверка
prompt = "The capital of Russia is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=5)
    print(f"Test generation: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")

/home/tikhonov/.conda/envs/sae_reasoning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer for unsloth/Meta-Llama-3.1-70B...
Loading model unsloth/Meta-Llama-3.1-70B in bfloat16...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 30/30 [14:33<00:00, 29.12s/it]


Done! Model is distributed across GPUs:
GPU 0
GPU 1
Test generation: The capital of Russia is a city of contrasts.
